# C2 · Tree instability: 30 resamples, 30 different trees

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Measure the **variance** of trees: resampling with the bootstrap, trees with the same parameters disagree on a large fraction of the examples. See how depth and minimum leaf size reduce the disagreement, and that the **majority vote** of 30 unlimited trees is much more accurate
than the single tree (the idea of forests).

## Dataset

`make_classification(1500, 10 features, 4 informative, flip_y=0.1)`, 60/40 split (`X_tr, y_tr, X_te, y_te`). For each configuration 30 trees are trained on bootstrap samples of the training set.

## Educational purpose

Turn "trees are unstable" into numbers: mean accuracy and variability of the single trees, fraction of examples on which two bootstrap trees disagree, and the variance reduction obtained by averaging. It is the direct motivation for the next module.

## Task

1. For `configs = {"no limits": {}, "max_depth=5": {"max_depth": 5}, "max_depth=3": {"max_depth": 3}, "min_samples_leaf=20": {"min_samples_leaf": 20}, "min_samples_leaf=50": {"min_samples_leaf": 50}}` train **30 trees** (`DecisionTreeClassifier(random_state=0, **params)`) on bootstrap samples (`rng = np.random.default_rng(0)`, indices `rng.integers(0, n, n)`) and collect the predictions on the test set.
2. Build `table` (rows: configurations; columns `mean_acc` (mean accuracy of the 30 trees), `acc_std` (standard deviation), `disagreement` (mean, over the pairs of the first 15 trees, of the fraction of test examples on which two trees give a different prediction), `mean_leaves`).
3. For the `no limits` configuration compute the **majority vote** of the 30 trees: `vote_acc` (accuracy of the vote) and `vote_gain = vote_acc − mean_acc` of the same configuration.
4. Save `disagreement_reduction`: `disagreement` of `no limits` divided by that of `max_depth=3`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(1500, n_features=10, n_informative=4, n_redundant=0, class_sep=0.8, flip_y=0.1, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
configs = {"no limits": {}, "max_depth=5": {"max_depth": 5}, "max_depth=3": {"max_depth": 3},
           "min_samples_leaf=20": {"min_samples_leaf": 20}, "min_samples_leaf=50": {"min_samples_leaf": 50}}

## Your work

In [ ]:
rng = np.random.default_rng(0)
predictions, leaves = {}, {}
for name, params in configs.items():
    ...

In [ ]:
table = pd.DataFrame(index=list(configs), columns=["mean_acc", "acc_std", "disagreement", "mean_leaves"], dtype=float)
table

In [ ]:
vote_acc = None
vote_gain = None
disagreement_reduction = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Loop: `P = []; F = []; for _ in range(30): i = rng.integers(0, len(X_tr), len(X_tr)); t = DecisionTreeClassifier(random_state=0, **params).fit(X_tr[i], y_tr[i]); P.append(t.predict(X_te)); F.append(t.get_n_leaves())`. Disagreement: `np.mean([(P[i] != P[j]).mean() for i in range(15) for j in range(i + 1, 15)])`. Vote: `(np.mean(P, axis=0) > 0.5).astype(int)`.'):
    check(set(predictions) == set(configs) and all(np.asarray(p).shape == (30, len(y_te)) for p in predictions.values()), "30 predictions on the test set for each configuration", "one row per tree, one column per test example")
    check(table.notna().all().all() and table.loc["no limits", "mean_acc"] < 0.76 and table.loc["max_depth=3", "mean_acc"] > 0.75, "the single trees without limits are below 0.76; with max_depth=3 above 0.75", "mean accuracy of the 30 trees")
    check(table.loc["no limits", "disagreement"] > 0.2 and table.loc["max_depth=3", "disagreement"] < 0.13 and table.loc["min_samples_leaf=50", "disagreement"] < 0.13, "without limits two trees disagree on over 20% of the examples; with max_depth=3 or leaf 50 on less than 13%", "mean over the pairs of trees")
    check(table.loc["no limits", "mean_leaves"] > 60 and table.loc["max_depth=3", "mean_leaves"] <= 8, "without limits over 60 leaves; max_depth=3 at most 8", "get_n_leaves()")
    check(vote_acc is not None and vote_acc > 0.78 and vote_gain > 0.04, "the vote of 30 trees without limits goes over 0.78: over 4 points above the single tree", "mean of the predictions > 0.5")
    check(disagreement_reduction > 1.8, "the disagreement without limits is over 1.8 times that of max_depth=3", "compare the two rows")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_tree_instability_and_the_vote.ipynb`. Open it after you pass the check, or after you have really tried.